# 04_bbox_annotation
프로젝트 루트에서 CLI로 실행 폴더를 준비한 뒤 MV_ANALYSIS_RUN 환경변수를 설정하세요. 01→06 순서입니다. 마스크 단계 전 mask-audit가 필요합니다.
입력: 실행 폴더의 manifest 및 REALIAD_DATA_ROOT. 예상 결과: defect_bboxes.csv. 알고리즘은 mv_analysis 모듈에 있습니다.
이미 계산된 결과를 읽어 검수합니다. 결과가 없으면 안내된 CLI를 터미널에서 실행하세요. 출력은 저장하지 않고 커밋 전 모두 지웁니다.

In [ ]:
import os, sys
from pathlib import Path
repo = Path.cwd() if (Path.cwd()/"mv_analysis").is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo))
from mv_analysis.cli import main
value = os.getenv("MV_ANALYSIS_RUN")
if not value: raise RuntimeError("먼저 MV_ANALYSIS_RUN=outputs/실행폴더를 설정한 뒤 Jupyter를 시작하세요")
run = Path(value) if Path(value).is_absolute() else repo / value
if not (run/"run.json").exists(): raise RuntimeError("실행 폴더에 run.json이 없습니다. CLI 실행부터 확인하세요")
result = run / "defect_bboxes.csv"
if not result.exists(): raise RuntimeError("결과 없음: 프로젝트 루트에서 --output 실행폴더 --resume bbox 실행. 원래 --limit/설정을 유지하세요")


In [ ]:
# 필요할 때만 실행: 기존 알고리즘을 복제하지 않고 CLI 모듈을 호출합니다.
RECOMPUTE = False
if RECOMPUTE:
    import json
    cfg = json.loads((run/"run.json").read_text())["config"]
    argv = ["--data-root", cfg["root"], "--output", str(run), "--resume", "bbox"]
    if cfg["limit"] is not None: argv += ["--limit", str(cfg["limit"])]
    if cfg["inventory_only"]: argv += ["--inventory-only"]
    argv += ["--class-policy", cfg["class_policy"]]
    if main(argv): raise RuntimeError("CLI 단계가 실패했습니다. 실행 로그를 확인하세요.")


In [ ]:
import pandas as pd
display(pd.read_csv(result).head(10))


In [ ]:
from IPython.display import Image, display
plot = run / "bbox_class_distribution.png"
if plot.exists(): display(Image(filename=str(plot), width=800))


## 검수 기록
표의 표본 수·품목·시점과 그림의 분포·예외 사례를 함께 확인하세요. 표본 결과를 전체 데이터로 일반화하지 마세요. 밝기는 조명/재질/결함이 섞여 있고, 위치 집중은 관측량 및 정렬에 좌우됩니다. 작은 박스와 해상도 손실은 실제 결함 crop과 연결해 확인해야 하며 모델 정확도를 뜻하지 않습니다.
실제 해석은 실행 폴더의 분석 보고서에서 읽고, 본인 검수 내용은 작은 텍스트로 별도 PR에 적으세요.